# 17 — Patient-Level Train / Validation / Test Split

This notebook builds a **leakage-safe, diagnosis-stratified, patient-level** split of the processed PANORAMA dataset.

It follows directly from `16_final_dataset_audit.ipynb`, which confirmed:

- 2,238 total cases, all with processed images and masks
- 2,224 unique patients, 11 with multiple studies (no diagnosis conflicts)
- Diagnosis distribution: 676 PDAC (30.21%) / 1,562 non-PDAC (69.79%)

**Splitting rule:** the split unit is `patient_id`, not `study_id`. A patient with multiple studies
must never appear in more than one split. Stratification is applied on diagnosis so each split
preserves, as closely as possible, the overall PDAC / non-PDAC ratio.

Do not run this notebook if `16_final_dataset_audit.ipynb` has not passed its final integrity check.

In [1]:
# ============================================================
# NOTEBOOK 17 — PATIENT-LEVEL DATASET SPLIT
# CELL 1 — IMPORTS AND CONFIGURATION
# ============================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"D:\Pancreatic_Cancer_Thesis"
)

DATA_DIR = PROJECT_ROOT / "data"

PROCESSED_DIR = (
    DATA_DIR / "processed"
)

METADATA_PATH = (
    PROCESSED_DIR / "metadata.csv"
)

SPLIT_OUTPUT_PATH = (
    PROCESSED_DIR / "split_assignment.csv"
)


# ------------------------------------------------------------
# Split configuration
# ------------------------------------------------------------

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

SPLIT_SEED = 42


# ------------------------------------------------------------
# Basic configuration checks
# ------------------------------------------------------------

assert np.isclose(
    TRAIN_FRAC + VAL_FRAC + TEST_FRAC,
    1.0,
)

assert 0 < TRAIN_FRAC < 1
assert 0 < VAL_FRAC < 1
assert 0 < TEST_FRAC < 1


print("=" * 70)
print("NOTEBOOK 17 — PATIENT-LEVEL DATASET SPLIT")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nMetadata:")
print(METADATA_PATH)

print("\nSplit output:")
print(SPLIT_OUTPUT_PATH)

print("\nSplit fractions:")
print("  Train      :", TRAIN_FRAC)
print("  Validation :", VAL_FRAC)
print("  Test       :", TEST_FRAC)

print("\nRandom seed:")
print(" ", SPLIT_SEED)

NOTEBOOK 17 — PATIENT-LEVEL DATASET SPLIT

Project root:
D:\Pancreatic_Cancer_Thesis

Metadata:
D:\Pancreatic_Cancer_Thesis\data\processed\metadata.csv

Split output:
D:\Pancreatic_Cancer_Thesis\data\processed\split_assignment.csv

Split fractions:
  Train      : 0.7
  Validation : 0.15
  Test       : 0.15

Random seed:
  42


In [2]:
# ============================================================
# CELL 2 — LOAD METADATA
# ============================================================

if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Metadata file not found:\n{METADATA_PATH}"
    )

metadata = pd.read_csv(
    METADATA_PATH
)

print("=" * 70)
print("METADATA LOADED")
print("=" * 70)

print("Rows    :", len(metadata))
print("Columns :", len(metadata.columns))

display(
    metadata.head()
)

METADATA LOADED
Rows    : 2238
Columns : 18


,study_id,label,image_path,mask_path,image_shape,mask_shape,image_dtype,mask_dtype,mask_labels,patient_id,patient_age,patient_sex,scanner,diagnosis,diagnosis_source,roi_size,target_spacing,hu_window
0,100000_00001,NaN,images\100000_00001.npy,masks\100000_00001.npy,"128,160,192","128,160,192",float32,uint8,"0,2,3,4,6",100000,42.0,F,TOSHIBA,non-PDAC,radiology,"128,160,192","1.0,1.0,3.0","-150,250"
1,100001_00001,NaN,images\100001_00001.npy,masks\100001_00001.npy,"128,160,192","128,160,192",float32,uint8,"0,2,3,4,5,6",100001,68.0,M,SIEMENS,non-PDAC,radiology,"128,160,192","1.0,1.0,3.0","-150,250"
2,100002_00001,NaN,images\100002_00001.npy,masks\100002_00001.npy,"128,160,192","128,160,192",float32,uint8,"0,1,2,3,4,5,6",100002,77.0,F,TOSHIBA,PDAC,pathology,"128,160,192","1.0,1.0,3.0","-150,250"
3,100003_00001,NaN,images\100003_00001.npy,masks\100003_00001.npy,"128,160,192","128,160,192",float32,uint8,"0,1,2,3,4,5,6",100003,57.0,M,TOSHIBA,PDAC,cytology,"128,160,192","1.0,1.0,3.0","-150,250"
4,100004_00001,NaN,images\100004_00001.npy,masks\100004_00001.npy,"128,160,192","128,160,192",float32,uint8,"0,2,3,4,5,6",100004,68.0,M,SIEMENS,non-PDAC,radiology,"128,160,192","1.0,1.0,3.0","-150,250"


In [3]:
# ============================================================
# CELL 3 — IDENTIFY REQUIRED COLUMNS
# ============================================================

def find_column(df, candidates):
    """
    Return the first matching column from candidates.
    Matching is case-insensitive.
    """
    lookup = {
        str(column).strip().lower(): column
        for column in df.columns
    }

    for candidate in candidates:
        key = candidate.strip().lower()

        if key in lookup:
            return lookup[key]

    return None


STUDY_ID_COL = find_column(
    metadata,
    [
        "study_id",
        "studyid",
        "case_id",
        "caseid",
    ],
)

PATIENT_ID_COL = find_column(
    metadata,
    [
        "patient_id",
        "patientid",
        "subject_id",
        "subjectid",
    ],
)

DIAGNOSIS_COL = find_column(
    metadata,
    [
        "diagnosis",
        "class",
        "target",
    ],
)

IMAGE_PATH_COL = find_column(
    metadata,
    [
        "image_path",
    ],
)

MASK_PATH_COL = find_column(
    metadata,
    [
        "mask_path",
    ],
)


print("=" * 70)
print("REQUIRED COLUMNS")
print("=" * 70)

print("Study ID   :", STUDY_ID_COL)
print("Patient ID :", PATIENT_ID_COL)
print("Diagnosis  :", DIAGNOSIS_COL)
print("Image path :", IMAGE_PATH_COL)
print("Mask path  :", MASK_PATH_COL)


required_columns = {
    "study_id": STUDY_ID_COL,
    "patient_id": PATIENT_ID_COL,
    "diagnosis": DIAGNOSIS_COL,
}

missing_required = [
    name
    for name, column in required_columns.items()
    if column is None
]

if missing_required:
    raise KeyError(
        "Missing required metadata columns: "
        + ", ".join(missing_required)
    )

print("\n✓ All required split columns are available.")

REQUIRED COLUMNS
Study ID   : study_id
Patient ID : patient_id
Diagnosis  : diagnosis
Image path : image_path
Mask path  : mask_path

✓ All required split columns are available.


In [4]:
# ============================================================
# CELL 4 — PRE-SPLIT VALIDATION
# ============================================================

print("=" * 70)
print("PRE-SPLIT VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# Normalize identifiers
# ------------------------------------------------------------

metadata = metadata.copy()

metadata[STUDY_ID_COL] = (
    metadata[STUDY_ID_COL]
    .astype(str)
    .str.strip()
)

metadata[PATIENT_ID_COL] = (
    metadata[PATIENT_ID_COL]
    .astype(str)
    .str.strip()
)

metadata[DIAGNOSIS_COL] = (
    metadata[DIAGNOSIS_COL]
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# Check missing identifiers / diagnosis
# ------------------------------------------------------------

missing_study_ids = (
    metadata[STUDY_ID_COL]
    .eq("")
    .sum()
)

missing_patient_ids = (
    metadata[PATIENT_ID_COL]
    .eq("")
    .sum()
)

missing_diagnosis = (
    metadata[DIAGNOSIS_COL]
    .eq("")
    .sum()
)

print("Missing study IDs    :", missing_study_ids)
print("Missing patient IDs  :", missing_patient_ids)
print("Missing diagnoses    :", missing_diagnosis)


# ------------------------------------------------------------
# Check study uniqueness
# ------------------------------------------------------------

duplicate_studies = (
    metadata[STUDY_ID_COL]
    .duplicated(keep=False)
)

print(
    "Duplicate study IDs  :",
    int(duplicate_studies.sum())
)


# ------------------------------------------------------------
# Diagnosis values
# ------------------------------------------------------------

valid_diagnoses = {
    "PDAC",
    "non-PDAC",
}

actual_diagnoses = set(
    metadata[DIAGNOSIS_COL]
    .dropna()
    .unique()
)

unexpected_diagnoses = (
    actual_diagnoses
    - valid_diagnoses
)

print(
    "Unexpected diagnoses :",
    unexpected_diagnoses
)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert missing_study_ids == 0
assert missing_patient_ids == 0
assert missing_diagnosis == 0
assert duplicate_studies.sum() == 0
assert not unexpected_diagnoses

print("\n✓ Pre-split validation passed.")

PRE-SPLIT VALIDATION
Missing study IDs    : 0
Missing patient IDs  : 0
Missing diagnoses    : 0
Duplicate study IDs  : 0
Unexpected diagnoses : set()

✓ Pre-split validation passed.


In [5]:
# ============================================================
# CELL 5 — BUILD PATIENT-LEVEL TABLE
# ============================================================

print("=" * 70)
print("PATIENT-LEVEL TABLE")
print("=" * 70)


# ------------------------------------------------------------
# Confirm each patient has one diagnosis
# ------------------------------------------------------------

diagnosis_counts_per_patient = (
    metadata
    .groupby(PATIENT_ID_COL)[DIAGNOSIS_COL]
    .nunique()
)

conflicting_patients = (
    diagnosis_counts_per_patient[
        diagnosis_counts_per_patient > 1
    ]
)

print(
    "Patients with conflicting diagnoses:",
    len(conflicting_patients)
)

assert len(conflicting_patients) == 0


# ------------------------------------------------------------
# Build patient-level table
# ------------------------------------------------------------

patient_level = (
    metadata
    .groupby(PATIENT_ID_COL)
    .agg(
        diagnosis=(
            DIAGNOSIS_COL,
            "first",
        ),
        case_count=(
            STUDY_ID_COL,
            "count",
        ),
    )
    .reset_index()
)

# Make the diagnosis column name explicit.
patient_level = patient_level.rename(
    columns={
        "diagnosis": "patient_diagnosis"
    }
)


print(
    "Unique patients:",
    len(patient_level)
)

print(
    "Total cases:",
    patient_level["case_count"].sum()
)

print("\nPatient diagnosis distribution:")
print(
    patient_level[
        "patient_diagnosis"
    ].value_counts()
)

print("\nCases per patient:")
print(
    patient_level[
        "case_count"
    ].value_counts()
    .sort_index()
)

display(
    patient_level.head(20)
)

PATIENT-LEVEL TABLE
Patients with conflicting diagnoses: 0
Unique patients: 2224
Total cases: 2238

Patient diagnosis distribution:
patient_diagnosis
non-PDAC    1551
PDAC         673
Name: count, dtype: int64

Cases per patient:
case_count
1    2213
2      10
5       1
Name: count, dtype: int64


,patient_id,patient_diagnosis,case_count
0,100000,non-PDAC,1
1,100001,non-PDAC,1
2,100002,PDAC,1
3,100003,PDAC,1
4,100004,non-PDAC,1
5,100005,PDAC,1
6,100006,non-PDAC,1
7,100007,non-PDAC,1
8,100008,non-PDAC,1
9,100009,PDAC,1


In [6]:
# ============================================================
# CELL 6 — PATIENT-LEVEL DIAGNOSIS DISTRIBUTION
# ============================================================

print("=" * 70)
print("PATIENT-LEVEL DIAGNOSIS DISTRIBUTION")
print("=" * 70)

patient_counts = (
    patient_level[
        "patient_diagnosis"
    ]
    .value_counts()
)

patient_percentages = (
    patient_level[
        "patient_diagnosis"
    ]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .round(2)
)

summary = pd.DataFrame({
    "patients": patient_counts,
    "percentage": patient_percentages,
})

display(summary)

print(
    "\nPDAC patients:",
    int(
        patient_counts.get(
            "PDAC",
            0
        )
    )
)

print(
    "Non-PDAC patients:",
    int(
        patient_counts.get(
            "non-PDAC",
            0
        )
    )
)

PATIENT-LEVEL DIAGNOSIS DISTRIBUTION


,patients,percentage
patient_diagnosis,,
non-PDAC,1551,69.74
PDAC,673,30.26



PDAC patients: 673
Non-PDAC patients: 1551


In [8]:
# ============================================================
# CELL 7 — PATIENT-LEVEL STRATIFIED SPLIT FUNCTION
# ============================================================

def create_patient_split(
    patient_df,
    patient_id_col,
    diagnosis_col,
    train_frac=0.70,
    val_frac=0.15,
    test_frac=0.15,
    seed=42,
):
    """
    Create a reproducible patient-level stratified split.

    Stratification is performed independently within each
    diagnosis group. The patient is the unit of splitting,
    not the individual CT study.
    """

    if not np.isclose(
        train_frac + val_frac + test_frac,
        1.0,
    ):
        raise ValueError(
            "Split fractions must sum to 1.0."
        )

    rng = np.random.default_rng(seed)

    assignments = {}

    for diagnosis in sorted(
        patient_df[diagnosis_col].unique()
    ):

        group = patient_df[
            patient_df[diagnosis_col]
            == diagnosis
        ].copy()

        patient_ids = (
            group[patient_id_col]
            .astype(str)
            .tolist()
        )

        patient_ids = np.array(
            patient_ids,
            dtype=object,
        )

        rng.shuffle(patient_ids)

        n = len(patient_ids)

        # Determine train and validation counts.
        # Test receives the remainder so every patient
        # is assigned exactly once.

        n_train = int(
            np.floor(
                n * train_frac
            )
        )

        n_val = int(
            np.floor(
                n * val_frac
            )
        )

        n_test = (
            n
            - n_train
            - n_val
        )

        train_ids = patient_ids[
            :n_train
        ]

        val_ids = patient_ids[
            n_train:n_train + n_val
        ]

        test_ids = patient_ids[
            n_train + n_val:
        ]

        for patient_id in train_ids:
            assignments[patient_id] = "train"

        for patient_id in val_ids:
            assignments[patient_id] = "validation"

        for patient_id in test_ids:
            assignments[patient_id] = "test"

    result = patient_df.copy()

    result["split"] = (
        result[patient_id_col]
        .astype(str)
        .map(assignments)
    )

    if result["split"].isna().any():
        raise RuntimeError(
            "Some patients were not assigned to a split."
        )

    return result

In [9]:
# ============================================================
# CELL 8 — CREATE PATIENT-LEVEL SPLIT
# ============================================================

patient_split = create_patient_split(
    patient_df=patient_level,
    patient_id_col=PATIENT_ID_COL,
    diagnosis_col="patient_diagnosis",
    train_frac=TRAIN_FRAC,
    val_frac=VAL_FRAC,
    test_frac=TEST_FRAC,
    seed=SPLIT_SEED,
)

print("=" * 70)
print("PATIENT-LEVEL SPLIT CREATED")
print("=" * 70)

print(
    patient_split["split"]
    .value_counts()
)

print("\nPatient-level diagnosis × split:")

display(
    pd.crosstab(
        patient_split["split"],
        patient_split["patient_diagnosis"],
        margins=True,
    )
)

PATIENT-LEVEL SPLIT CREATED
split
train         1556
test           336
validation     332
Name: count, dtype: int64

Patient-level diagnosis × split:


patient_diagnosis,PDAC,non-PDAC,All
split,,,
test,102,234,336
train,471,1085,1556
validation,100,232,332
All,673,1551,2224


In [10]:
# ============================================================
# CELL 9 — MAP PATIENT SPLIT TO STUDIES
# ============================================================

print("=" * 70)
print("MAPPING PATIENT SPLITS TO STUDIES")
print("=" * 70)

patient_assignment = (
    patient_split[
        [
            PATIENT_ID_COL,
            "patient_diagnosis",
            "split",
        ]
    ]
)

split_df = metadata.merge(
    patient_assignment,
    on=PATIENT_ID_COL,
    how="left",
    validate="many_to_one",
)

print(
    "Metadata cases:",
    len(metadata)
)

print(
    "Assigned cases:",
    len(split_df)
)

assert len(split_df) == len(metadata)

assert (
    split_df["split"]
    .notna()
    .all()
)

print("\nCase counts:")
print(
    split_df["split"]
    .value_counts()
)

print("\nCase diagnosis × split:")

display(
    pd.crosstab(
        split_df["split"],
        split_df[DIAGNOSIS_COL],
        margins=True,
    )
)

MAPPING PATIENT SPLITS TO STUDIES
Metadata cases: 2238
Assigned cases: 2238

Case counts:
split
train         1563
validation     338
test           337
Name: count, dtype: int64

Case diagnosis × split:


diagnosis,PDAC,non-PDAC,All
split,,,
test,103,234,337
train,473,1090,1563
validation,100,238,338
All,676,1562,2238


In [11]:
# ============================================================
# CELL 10 — PATIENT LEAKAGE CHECK
# ============================================================

print("=" * 70)
print("PATIENT LEAKAGE VERIFICATION")
print("=" * 70)

patient_split_counts = (
    split_df
    .groupby(PATIENT_ID_COL)["split"]
    .nunique()
)

leaking_patients = (
    patient_split_counts[
        patient_split_counts > 1
    ]
)

print(
    "Total patients:",
    patient_split_counts.shape[0]
)

print(
    "Patients in multiple splits:",
    len(leaking_patients)
)

if len(leaking_patients) > 0:

    print("\nLeaking patients:")
    display(
        split_df[
            split_df[PATIENT_ID_COL].isin(
                leaking_patients.index
            )
        ]
        .sort_values(PATIENT_ID_COL)
    )

    raise AssertionError(
        "Patient leakage detected."
    )

print(
    "\n✓ No patient appears in more than one split."
)

PATIENT LEAKAGE VERIFICATION
Total patients: 2224
Patients in multiple splits: 0

✓ No patient appears in more than one split.


In [12]:
# ============================================================
# CELL 11 — MULTI-CASE PATIENT SPOT CHECK
# ============================================================

print("=" * 70)
print("MULTI-CASE PATIENT SPOT CHECK")
print("=" * 70)

multi_case_ids = (
    patient_level[
        patient_level["case_count"] > 1
    ][PATIENT_ID_COL]
    .tolist()
)

multi_case_assignments = (
    split_df[
        split_df[PATIENT_ID_COL].isin(
            multi_case_ids
        )
    ]
    [
        [
            PATIENT_ID_COL,
            STUDY_ID_COL,
            DIAGNOSIS_COL,
            "split",
        ]
    ]
    .sort_values(
        [
            PATIENT_ID_COL,
            STUDY_ID_COL,
        ]
    )
)

display(
    multi_case_assignments
)

# Confirm each patient has one split
assert (
    multi_case_assignments
    .groupby(PATIENT_ID_COL)["split"]
    .nunique()
    .max()
    == 1
)

print(
    "\n✓ All multi-case patients remain entirely within one split."
)

MULTI-CASE PATIENT SPOT CHECK


,patient_id,study_id,diagnosis,split
47,100047,100047_00001,non-PDAC,validation
48,100047,100047_00002,non-PDAC,validation
49,100047,100047_00003,non-PDAC,validation
50,100047,100047_00004,non-PDAC,validation
51,100047,100047_00005,non-PDAC,validation
269,100265,100265_00001,PDAC,train
270,100265,100265_00002,PDAC,train
273,100268,100268_00001,non-PDAC,train
274,100268,100268_00002,non-PDAC,train
422,100416,100416_00001,PDAC,train



✓ All multi-case patients remain entirely within one split.


In [13]:
# ============================================================
# CELL 12 — STUDY ASSIGNMENT INTEGRITY
# ============================================================

print("=" * 70)
print("STUDY ASSIGNMENT INTEGRITY")
print("=" * 70)

print(
    "Metadata cases:",
    len(metadata)
)

print(
    "Split rows:",
    len(split_df)
)

print(
    "Unique study IDs in split:",
    split_df[STUDY_ID_COL].nunique()
)

print(
    "Unique study IDs in metadata:",
    metadata[STUDY_ID_COL].nunique()
)

assert len(split_df) == len(metadata)

assert (
    split_df[STUDY_ID_COL].nunique()
    == metadata[STUDY_ID_COL].nunique()
)

assert (
    split_df[STUDY_ID_COL].is_unique
)

assert (
    split_df["split"].isin(
        [
            "train",
            "validation",
            "test",
        ]
    ).all()
)

print(
    "\n✓ Every study is assigned to exactly one split."
)

STUDY ASSIGNMENT INTEGRITY
Metadata cases: 2238
Split rows: 2238
Unique study IDs in split: 2238
Unique study IDs in metadata: 2238

✓ Every study is assigned to exactly one split.


In [14]:
# ============================================================
# CELL 13 — PATIENT-LEVEL SPLIT SUMMARY
# ============================================================

print("=" * 70)
print("PATIENT-LEVEL SPLIT SUMMARY")
print("=" * 70)

patient_summary = (
    patient_split
    .groupby("split")
    .agg(
        patients=(
            PATIENT_ID_COL,
            "nunique",
        ),
        pdac_patients=(
            "patient_diagnosis",
            lambda s: (
                s == "PDAC"
            ).sum()
        ),
        non_pdac_patients=(
            "patient_diagnosis",
            lambda s: (
                s == "non-PDAC"
            ).sum()
        ),
    )
)

patient_summary[
    "pdac_percentage"
] = (
    patient_summary[
        "pdac_patients"
    ]
    / patient_summary["patients"]
    * 100
).round(2)

display(
    patient_summary
)

PATIENT-LEVEL SPLIT SUMMARY


,patients,pdac_patients,non_pdac_patients,pdac_percentage
split,,,,
test,336,102,234,30.36
train,1556,471,1085,30.27
validation,332,100,232,30.12


In [15]:
# ============================================================
# CELL 14 — CASE-LEVEL SPLIT SUMMARY
# ============================================================

print("=" * 70)
print("CASE-LEVEL SPLIT SUMMARY")
print("=" * 70)

case_summary = (
    split_df
    .groupby("split")
    .agg(
        cases=(
            STUDY_ID_COL,
            "nunique",
        ),
        pdac_cases=(
            DIAGNOSIS_COL,
            lambda s: (
                s == "PDAC"
            ).sum()
        ),
        non_pdac_cases=(
            DIAGNOSIS_COL,
            lambda s: (
                s == "non-PDAC"
            ).sum()
        ),
    )
)

case_summary[
    "pdac_percentage"
] = (
    case_summary[
        "pdac_cases"
    ]
    / case_summary["cases"]
    * 100
).round(2)

display(
    case_summary
)

CASE-LEVEL SPLIT SUMMARY


,cases,pdac_cases,non_pdac_cases,pdac_percentage
split,,,,
test,337,103,234,30.56
train,1563,473,1090,30.26
validation,338,100,238,29.59


In [16]:
# ============================================================
# CELL 15 — STRATIFICATION QUALITY
# ============================================================

print("=" * 70)
print("STRATIFICATION QUALITY")
print("=" * 70)

overall_pdac_percentage = (
    (
        metadata[DIAGNOSIS_COL]
        == "PDAC"
    ).mean()
    * 100
)

print(
    "Overall PDAC case prevalence:",
    round(
        overall_pdac_percentage,
        2,
    ),
    "%"
)

comparison = case_summary[
    ["cases", "pdac_cases", "non_pdac_cases", "pdac_percentage"]
].copy()

comparison[
    "difference_from_overall_pp"
] = (
    comparison["pdac_percentage"]
    - overall_pdac_percentage
).round(2)

display(comparison)

print(
    "\nMaximum absolute difference:",
    round(
        comparison[
            "difference_from_overall_pp"
        ]
        .abs()
        .max(),
        2,
    ),
    "percentage points",
)

STRATIFICATION QUALITY
Overall PDAC case prevalence: 30.21 %


,cases,pdac_cases,non_pdac_cases,pdac_percentage,difference_from_overall_pp
split,,,,,
test,337,103,234,30.56,0.35
train,1563,473,1090,30.26,0.05
validation,338,100,238,29.59,-0.62



Maximum absolute difference: 0.62 percentage points


In [17]:
# ============================================================
# CELL 16 — DETERMINISTIC REPRODUCIBILITY CHECK
# ============================================================

print("=" * 70)
print("DETERMINISTIC SPLIT CHECK")
print("=" * 70)

patient_split_repeat = create_patient_split(
    patient_df=patient_level,
    patient_id_col=PATIENT_ID_COL,
    diagnosis_col="patient_diagnosis",
    train_frac=TRAIN_FRAC,
    val_frac=VAL_FRAC,
    test_frac=TEST_FRAC,
    seed=SPLIT_SEED,
)

first_assignment = (
    patient_split[
        [
            PATIENT_ID_COL,
            "split",
        ]
    ]
    .sort_values(PATIENT_ID_COL)
    .reset_index(drop=True)
)

repeat_assignment = (
    patient_split_repeat[
        [
            PATIENT_ID_COL,
            "split",
        ]
    ]
    .sort_values(PATIENT_ID_COL)
    .reset_index(drop=True)
)

deterministic = (
    first_assignment
    .equals(repeat_assignment)
)

print(
    "Same seed:",
    SPLIT_SEED
)

print(
    "Assignments identical:",
    deterministic
)

assert deterministic

print(
    "\n✓ Split is deterministic and reproducible."
)

DETERMINISTIC SPLIT CHECK
Same seed: 42
Assignments identical: True

✓ Split is deterministic and reproducible.


In [18]:
# ============================================================
# CELL 17 — BUILD SPLIT ASSIGNMENT FILE
# ============================================================

print("=" * 70)
print("BUILDING SPLIT ASSIGNMENT FILE")
print("=" * 70)

split_columns = [
    STUDY_ID_COL,
    PATIENT_ID_COL,
    DIAGNOSIS_COL,
]

if IMAGE_PATH_COL is not None:
    split_columns.append(IMAGE_PATH_COL)

if MASK_PATH_COL is not None:
    split_columns.append(MASK_PATH_COL)

split_columns.append("split")

split_assignment = (
    split_df[split_columns]
    .copy()
    .sort_values(STUDY_ID_COL)
    .reset_index(drop=True)
)

print(
    "Rows:",
    len(split_assignment)
)

print(
    "Columns:",
    list(split_assignment.columns)
)

display(
    split_assignment.head(10)
)

BUILDING SPLIT ASSIGNMENT FILE
Rows: 2238
Columns: ['study_id', 'patient_id', 'diagnosis', 'image_path', 'mask_path', 'split']


,study_id,patient_id,diagnosis,image_path,mask_path,split
0,100000_00001,100000,non-PDAC,images\100000_00001.npy,masks\100000_00001.npy,train
1,100001_00001,100001,non-PDAC,images\100001_00001.npy,masks\100001_00001.npy,train
2,100002_00001,100002,PDAC,images\100002_00001.npy,masks\100002_00001.npy,train
3,100003_00001,100003,PDAC,images\100003_00001.npy,masks\100003_00001.npy,test
4,100004_00001,100004,non-PDAC,images\100004_00001.npy,masks\100004_00001.npy,train
5,100005_00001,100005,PDAC,images\100005_00001.npy,masks\100005_00001.npy,train
6,100006_00001,100006,non-PDAC,images\100006_00001.npy,masks\100006_00001.npy,test
7,100007_00001,100007,non-PDAC,images\100007_00001.npy,masks\100007_00001.npy,train
8,100008_00001,100008,non-PDAC,images\100008_00001.npy,masks\100008_00001.npy,train
9,100009_00001,100009,PDAC,images\100009_00001.npy,masks\100009_00001.npy,train


In [19]:
# ============================================================
# CELL 18 — FINAL SPLIT FILE VALIDATION
# ============================================================

print("=" * 70)
print("FINAL SPLIT FILE VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# Row count
# ------------------------------------------------------------

assert (
    len(split_assignment)
    == len(metadata)
)

# ------------------------------------------------------------
# Study uniqueness
# ------------------------------------------------------------

assert (
    split_assignment[
        STUDY_ID_COL
    ].is_unique
)

# ------------------------------------------------------------
# Split values
# ------------------------------------------------------------

valid_splits = {
    "train",
    "validation",
    "test",
}

actual_splits = set(
    split_assignment["split"]
)

assert actual_splits == valid_splits


# ------------------------------------------------------------
# Patient leakage
# ------------------------------------------------------------

split_counts = (
    split_assignment
    .groupby(PATIENT_ID_COL)["split"]
    .nunique()
)

assert (
    split_counts.max()
    == 1
)


# ------------------------------------------------------------
# All patients represented
# ------------------------------------------------------------

assert (
    split_assignment[PATIENT_ID_COL]
    .nunique()
    == patient_level[PATIENT_ID_COL]
    .nunique()
)


print("Rows:", len(split_assignment))

print(
    "Unique studies:",
    split_assignment[
        STUDY_ID_COL
    ].nunique()
)

print(
    "Unique patients:",
    split_assignment[
        PATIENT_ID_COL
    ].nunique()
)

print(
    "Train cases:",
    int(
        (
            split_assignment["split"]
            == "train"
        ).sum()
    )
)

print(
    "Validation cases:",
    int(
        (
            split_assignment["split"]
            == "validation"
        ).sum()
    )
)

print(
    "Test cases:",
    int(
        (
            split_assignment["split"]
            == "test"
        ).sum()
    )
)

print(
    "\nPatients appearing in multiple splits:",
    int(
        (
            split_counts > 1
        ).sum()
    )
)

print(
    "\n✓ Final split assignment passed validation."
)

FINAL SPLIT FILE VALIDATION
Rows: 2238
Unique studies: 2238
Unique patients: 2224
Train cases: 1563
Validation cases: 338
Test cases: 337

Patients appearing in multiple splits: 0

✓ Final split assignment passed validation.


In [20]:
# ============================================================
# CELL 19 — SAVE SPLIT ASSIGNMENT
# ============================================================

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

split_assignment.to_csv(
    SPLIT_OUTPUT_PATH,
    index=False,
)

print("=" * 70)
print("SPLIT ASSIGNMENT SAVED")
print("=" * 70)

print(
    "Path:",
    SPLIT_OUTPUT_PATH
)

print(
    "Rows:",
    len(split_assignment)
)

print(
    "Size:",
    SPLIT_OUTPUT_PATH.stat().st_size,
    "bytes"
)

SPLIT ASSIGNMENT SAVED
Path: D:\Pancreatic_Cancer_Thesis\data\processed\split_assignment.csv
Rows: 2238
Size: 184461 bytes


In [21]:
# ============================================================
# CELL 20 — READ-BACK VERIFICATION
# ============================================================

print("=" * 70)
print("READ-BACK VERIFICATION")
print("=" * 70)

if not SPLIT_OUTPUT_PATH.exists():
    raise FileNotFoundError(
        SPLIT_OUTPUT_PATH
    )

saved_split = pd.read_csv(
    SPLIT_OUTPUT_PATH
)

print(
    "Saved rows:",
    len(saved_split)
)

print(
    "Saved columns:",
    list(saved_split.columns)
)

assert (
    len(saved_split)
    == len(metadata)
)

assert (
    saved_split[STUDY_ID_COL]
    .is_unique
)

assert (
    saved_split["split"]
    .isin(
        [
            "train",
            "validation",
            "test",
        ]
    )
    .all()
)

saved_patient_split_counts = (
    saved_split
    .groupby(PATIENT_ID_COL)["split"]
    .nunique()
)

assert (
    saved_patient_split_counts.max()
    == 1
)

print(
    "Unique studies:",
    saved_split[
        STUDY_ID_COL
    ].nunique()
)

print(
    "Unique patients:",
    saved_split[
        PATIENT_ID_COL
    ].nunique()
)

print(
    "Patients in multiple splits:",
    int(
        (
            saved_patient_split_counts
            > 1
        ).sum()
    )
)

print(
    "\n✓ Saved split file verified."
)

READ-BACK VERIFICATION
Saved rows: 2238
Saved columns: ['study_id', 'patient_id', 'diagnosis', 'image_path', 'mask_path', 'split']
Unique studies: 2238
Unique patients: 2224
Patients in multiple splits: 0

✓ Saved split file verified.


In [22]:
# ============================================================
# CELL 21 — FINAL SPLIT REPORT
# ============================================================

print("=" * 70)
print("FINAL PATIENT-LEVEL SPLIT REPORT")
print("=" * 70)

print("\nDataset")
print("  Total cases        :", len(metadata))
print(
    "  Unique patients    :",
    metadata[PATIENT_ID_COL].nunique()
)
print(
    "  Unique study IDs   :",
    metadata[STUDY_ID_COL].nunique()
)

print("\nSplit configuration")
print("  Train fraction     :", TRAIN_FRAC)
print("  Validation fraction:", VAL_FRAC)
print("  Test fraction      :", TEST_FRAC)
print("  Random seed        :", SPLIT_SEED)

print("\nCase counts")
for split_name in [
    "train",
    "validation",
    "test",
]:
    subset = saved_split[
        saved_split["split"]
        == split_name
    ]

    pdac = int(
        (
            subset[DIAGNOSIS_COL]
            == "PDAC"
        ).sum()
    )

    non_pdac = int(
        (
            subset[DIAGNOSIS_COL]
            == "non-PDAC"
        ).sum()
    )

    print(
        f"  {split_name:<12}: "
        f"{len(subset):4d} cases | "
        f"PDAC {pdac:4d} | "
        f"non-PDAC {non_pdac:4d}"
    )


print("\nPatient leakage")
print(
    "  Patients in multiple splits:",
    int(
        (
            saved_patient_split_counts
            > 1
        ).sum()
    )
)

print("\n" + "-" * 70)

assert (
    len(saved_split)
    == len(metadata)
)

assert (
    saved_split[STUDY_ID_COL]
    .is_unique
)

assert (
    saved_patient_split_counts.max()
    == 1
)

print(
    "✓ FINAL SPLIT PASSED ALL CHECKS"
)

print(
    "✓ Split assignment is reproducible."
)

print(
    "✓ No patient leakage detected."
)

print(
    "✓ All 2,238 cases are assigned exactly once."
)

print(
    "\nSplit file:"
)

print(
    SPLIT_OUTPUT_PATH
)

FINAL PATIENT-LEVEL SPLIT REPORT

Dataset
  Total cases        : 2238
  Unique patients    : 2224
  Unique study IDs   : 2238

Split configuration
  Train fraction     : 0.7
  Validation fraction: 0.15
  Test fraction      : 0.15
  Random seed        : 42

Case counts
  train       : 1563 cases | PDAC  473 | non-PDAC 1090
  validation  :  338 cases | PDAC  100 | non-PDAC  238
  test        :  337 cases | PDAC  103 | non-PDAC  234

Patient leakage
  Patients in multiple splits: 0

----------------------------------------------------------------------
✓ FINAL SPLIT PASSED ALL CHECKS
✓ Split assignment is reproducible.
✓ No patient leakage detected.
✓ All 2,238 cases are assigned exactly once.

Split file:
D:\Pancreatic_Cancer_Thesis\data\processed\split_assignment.csv
